# 01 — Data Exploration & Quality Assessment

**Goal:** Load the Crunchbase 2015 relational dataset, assess quality, validate joins, and confirm feasibility for temporal graph construction.

**Dataset:** `data/raw/crunchbase_2015/` (investments.csv, rounds.csv, companies.csv, acquisitions.csv)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

DATA_DIR = Path("../data/raw/crunchbase_2015")

## 1. Load Raw Data

In [ ]:
companies = pd.read_csv(DATA_DIR / "companies.csv")
rounds = pd.read_csv(DATA_DIR / "rounds.csv")
investments = pd.read_csv(DATA_DIR / "investments.csv")
acquisitions = pd.read_csv(DATA_DIR / "acquisitions.csv")

print(f"Companies:    {companies.shape[0]:>8,} rows x {companies.shape[1]} cols")
print(f"Rounds:       {rounds.shape[0]:>8,} rows x {rounds.shape[1]} cols")
print(f"Investments:  {investments.shape[0]:>8,} rows x {investments.shape[1]} cols")
print(f"Acquisitions: {acquisitions.shape[0]:>8,} rows x {acquisitions.shape[1]} cols")

## 2. Schema Inspection

In [ ]:
print("=== COMPANIES ===")
print(companies.dtypes)
print(f"\nMissing values:\n{companies.isnull().sum()}")
companies.head(3)

In [ ]:
print("=== ROUNDS ===")
print(rounds.dtypes)
print(f"\nMissing values:\n{rounds.isnull().sum()}")
rounds.head(3)

In [ ]:
print("=== INVESTMENTS ===")
print(investments.dtypes)
print(f"\nMissing values:\n{investments.isnull().sum()}")
investments.head(3)

## 3. Date Parsing & Temporal Coverage

In [ ]:
# Parse dates
rounds['funded_at'] = pd.to_datetime(rounds['funded_at'], errors='coerce')
investments['funded_at'] = pd.to_datetime(investments['funded_at'], errors='coerce')
companies['founded_at'] = pd.to_datetime(companies['founded_at'], errors='coerce')
companies['first_funding_at'] = pd.to_datetime(companies['first_funding_at'], errors='coerce')
companies['last_funding_at'] = pd.to_datetime(companies['last_funding_at'], errors='coerce')

# Date parsing success rate
rounds_date_valid = rounds['funded_at'].notna().sum()
inv_date_valid = investments['funded_at'].notna().sum()

print(f"Rounds: {rounds_date_valid}/{len(rounds)} dates parsed ({100*rounds_date_valid/len(rounds):.1f}%)")
print(f"Investments: {inv_date_valid}/{len(investments)} dates parsed ({100*inv_date_valid/len(investments):.1f}%)")
print(f"\nRounds date range: {rounds['funded_at'].min()} to {rounds['funded_at'].max()}")
print(f"Investments date range: {investments['funded_at'].min()} to {investments['funded_at'].max()}")

In [ ]:
# Temporal distribution
rounds_by_year = rounds['funded_at'].dt.year.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
rounds_by_year[rounds_by_year.index >= 2000].plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Funding Rounds by Year (2000+)')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Rounds')

# Mark our temporal splits
ax.axvline(x=list(rounds_by_year[rounds_by_year.index >= 2000].index).index(2013), 
           color='red', linestyle='--', label='Train/Val split (2013)')
ax.axvline(x=list(rounds_by_year[rounds_by_year.index >= 2000].index).index(2014),
           color='orange', linestyle='--', label='Val/Test split (2014)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Entity Statistics

In [ ]:
print("=== UNIQUE ENTITIES ===")
print(f"Companies in companies.csv:    {companies['permalink'].nunique():,}")
print(f"Companies in rounds.csv:       {rounds['company_permalink'].nunique():,}")
print(f"Companies in investments.csv:  {investments['company_permalink'].nunique():,}")
print(f"Investors in investments.csv:  {investments['investor_permalink'].nunique():,}")
print(f"Unique rounds in rounds.csv:   {rounds['funding_round_permalink'].nunique():,}")

# Join coverage
companies_in_rounds = set(rounds['company_permalink'].unique())
companies_in_investments = set(investments['company_permalink'].unique())
companies_master = set(companies['permalink'].unique())

print(f"\n=== JOIN COVERAGE ===")
print(f"Companies with rounds data: {len(companies_in_rounds & companies_master):,} / {len(companies_master):,}")
print(f"Companies with investment edges: {len(companies_in_investments & companies_master):,} / {len(companies_master):,}")
print(f"Companies in rounds but NOT in master: {len(companies_in_rounds - companies_master):,}")

## 5. Round Type Distribution

In [ ]:
print("=== ROUND TYPES ===")
print(rounds['funding_round_type'].value_counts())
print(f"\n=== ROUND CODES (Series A, B, C...) ===")
print(rounds['funding_round_code'].value_counts().head(10))

In [ ]:
# Identify our target population: Seed and Series A rounds
seed_rounds = rounds[rounds['funding_round_type'] == 'seed']
series_a_rounds = rounds[
    (rounds['funding_round_type'] == 'venture') & 
    (rounds['funding_round_code'] == 'A')
]

print(f"Seed rounds: {len(seed_rounds):,}")
print(f"Series A rounds: {len(series_a_rounds):,}")
print(f"Total eligible trigger rounds: {len(seed_rounds) + len(series_a_rounds):,}")
print(f"\nUnique companies with Seed: {seed_rounds['company_permalink'].nunique():,}")
print(f"Unique companies with Series A: {series_a_rounds['company_permalink'].nunique():,}")

## 6. Multi-Round Companies (Label Feasibility)

In [ ]:
# How many companies have multiple rounds?
rounds_per_company = rounds.groupby('company_permalink').size()

print("=== ROUNDS PER COMPANY ===")
print(rounds_per_company.describe())
print(f"\nCompanies with 1 round:  {(rounds_per_company == 1).sum():,}")
print(f"Companies with 2 rounds: {(rounds_per_company == 2).sum():,}")
print(f"Companies with 3+ rounds: {(rounds_per_company >= 3).sum():,}")
print(f"Companies with 5+ rounds: {(rounds_per_company >= 5).sum():,}")

In [ ]:
# Estimate follow-on rate for Seed companies
seed_companies = set(seed_rounds['company_permalink'].unique())
companies_with_post_seed = set(
    rounds[
        (rounds['company_permalink'].isin(seed_companies)) &
        (rounds['funding_round_type'] != 'seed')
    ]['company_permalink'].unique()
)

print(f"Companies with Seed round: {len(seed_companies):,}")
print(f"Of those, raised non-Seed follow-on: {len(companies_with_post_seed):,}")
print(f"Raw follow-on rate: {100*len(companies_with_post_seed)/len(seed_companies):.1f}%")
print("\n(Note: This is raw rate without 24-month window constraint)")

## 7. Graph Density Assessment

In [ ]:
# Investor portfolio sizes
investor_portfolio_size = investments.groupby('investor_permalink')['company_permalink'].nunique()

print("=== INVESTOR PORTFOLIO SIZES ===")
print(investor_portfolio_size.describe())
print(f"\nInvestors with 1 company: {(investor_portfolio_size == 1).sum():,}")
print(f"Investors with 5+ companies: {(investor_portfolio_size >= 5).sum():,}")
print(f"Investors with 20+ companies: {(investor_portfolio_size >= 20).sum():,}")
print(f"Investors with 100+ companies: {(investor_portfolio_size >= 100).sum():,}")

In [ ]:
# Startup investor counts
startup_investor_count = investments.groupby('company_permalink')['investor_permalink'].nunique()

print("=== INVESTORS PER STARTUP ===")
print(startup_investor_count.describe())
print(f"\nStartups with 1 investor: {(startup_investor_count == 1).sum():,}")
print(f"Startups with 3+ investors: {(startup_investor_count >= 3).sum():,}")
print(f"Startups with 5+ investors: {(startup_investor_count >= 5).sum():,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Investor portfolio size distribution (log scale)
axes[0].hist(investor_portfolio_size.clip(upper=50), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Investor Portfolio Size Distribution')
axes[0].set_xlabel('Number of Portfolio Companies (clipped at 50)')
axes[0].set_ylabel('Count')

# Startup investor count distribution
axes[1].hist(startup_investor_count.clip(upper=20), bins=20, color='coral', edgecolor='white')
axes[1].set_title('Investors per Startup Distribution')
axes[1].set_xlabel('Number of Investors (clipped at 20)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 8. Geographic & Category Distribution

In [ ]:
print("=== TOP COUNTRIES ===")
print(companies['country_code'].value_counts().head(15))
print(f"\n=== TOP CATEGORIES ===")
# Categories are pipe-delimited
all_categories = companies['category_list'].dropna().str.split('|').explode()
print(all_categories.value_counts().head(20))

In [ ]:
# Company status distribution
print("=== COMPANY STATUS ===")
print(companies['status'].value_counts())

## 9. Funding Amount Analysis

In [ ]:
# Parse amounts
rounds['raised_amount_usd'] = pd.to_numeric(rounds['raised_amount_usd'], errors='coerce')

print("=== AMOUNT BY ROUND TYPE (USD) ===")
amount_by_type = rounds.groupby('funding_round_type')['raised_amount_usd'].describe()
print(amount_by_type[['count', 'mean', 'median', 'max']].sort_values('count', ascending=False).head(10))

# How many rounds have amount data?
amount_valid = rounds['raised_amount_usd'].notna().sum()
print(f"\nRounds with amount data: {amount_valid:,} / {len(rounds):,} ({100*amount_valid/len(rounds):.1f}%)")

## 10. Temporal Split Preview

In [ ]:
# Preview what our temporal splits look like
rounds_valid = rounds[rounds['funded_at'].notna()].copy()

train_rounds = rounds_valid[rounds_valid['funded_at'] < '2013-01-01']
val_rounds = rounds_valid[
    (rounds_valid['funded_at'] >= '2013-01-01') & 
    (rounds_valid['funded_at'] < '2014-01-01')
]
test_rounds = rounds_valid[
    (rounds_valid['funded_at'] >= '2014-01-01') & 
    (rounds_valid['funded_at'] < '2014-07-01')
]

print("=== TEMPORAL SPLIT PREVIEW (All rounds) ===")
print(f"Train (< 2013):         {len(train_rounds):>7,} rounds | {train_rounds['company_permalink'].nunique():>6,} companies")
print(f"Val (2013):             {len(val_rounds):>7,} rounds | {val_rounds['company_permalink'].nunique():>6,} companies")
print(f"Test (2014-H1):         {len(test_rounds):>7,} rounds | {test_rounds['company_permalink'].nunique():>6,} companies")

# Now filter to only Seed/Series A triggers
def is_trigger(row):
    return (row['funding_round_type'] == 'seed') or \
           (row['funding_round_type'] == 'venture' and row['funding_round_code'] == 'A')

train_triggers = train_rounds[train_rounds.apply(is_trigger, axis=1)]
val_triggers = val_rounds[val_rounds.apply(is_trigger, axis=1)]
test_triggers = test_rounds[test_rounds.apply(is_trigger, axis=1)]

print(f"\n=== TRIGGER ROUNDS ONLY (Seed + Series A) ===")
print(f"Train triggers: {len(train_triggers):>6,} | {train_triggers['company_permalink'].nunique():>5,} unique companies")
print(f"Val triggers:   {len(val_triggers):>6,} | {val_triggers['company_permalink'].nunique():>5,} unique companies")
print(f"Test triggers:  {len(test_triggers):>6,} | {test_triggers['company_permalink'].nunique():>5,} unique companies")

## 11. Summary & Key Findings

**Record this section after running all cells above.**

In [ ]:
print("="*60)
print("EXPLORATION SUMMARY")
print("="*60)
print(f"\nDataset viability: CONFIRMED")
print(f"\nKey statistics:")
print(f"  • Companies: {companies.shape[0]:,}")
print(f"  • Investment edges: {investments.shape[0]:,}")
print(f"  • Unique investors: {investments['investor_permalink'].nunique():,}")
print(f"  • Funding rounds: {rounds.shape[0]:,}")
print(f"  • Date coverage: {rounds['funded_at'].min().year}–{rounds['funded_at'].max().year}")
print(f"  • Date parse rate: {100*rounds_date_valid/len(rounds):.1f}%")
print(f"\nLabel generation feasibility:")
print(f"  • Seed companies: {len(seed_companies):,}")
print(f"  • Series A companies: {series_a_rounds['company_permalink'].nunique():,}")
print(f"  • Raw follow-on rate (Seed): {100*len(companies_with_post_seed)/len(seed_companies):.1f}%")
print(f"\nGraph density:")
print(f"  • Median investors per startup: {startup_investor_count.median():.0f}")
print(f"  • Median portfolio size: {investor_portfolio_size.median():.0f}")
print(f"  • Investors with 5+ companies: {(investor_portfolio_size >= 5).sum():,}")
print(f"\n✅ Proceed to Step 2: Label Generation")